In [1]:
import sys
import os
from tqdm import tqdm
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
from sklearn.model_selection import train_test_split

# -----------------------------
# Load Cleaned Data
# -----------------------------
CLEANED_DATA_PATH = '../data/processed/filtered_complaints.csv'
if not os.path.exists(CLEANED_DATA_PATH):
    raise FileNotFoundError(f"❌ Cleaned data not found at {CLEANED_DATA_PATH}")

df = pd.read_csv(CLEANED_DATA_PATH)
print(f"✅ Cleaned dataset loaded. Shape: {df.shape}")

# -----------------------------
# Stratified Sampling
# -----------------------------
sample_size = 15000

if sample_size > len(df):
    sample_size = len(df)
    print(f"⚠ Sample size larger than dataset. Using full dataset: {sample_size} records")

df_sample, _ = train_test_split(
    df,
    train_size=sample_size,
    stratify=df['Product'],
    random_state=42
)

print(f"Sampled Dataset Shape: {df_sample.shape}")
print("Product Distribution in Sample:\n", df_sample['Product'].value_counts(normalize=True))


✅ Cleaned dataset loaded. Shape: (426841, 20)
Sampled Dataset Shape: (15000, 20)
Product Distribution in Sample:
 Product
Checking or savings account                           0.328733
Credit card or prepaid card                           0.254600
Money transfer, virtual currency, or money service    0.227667
Credit card                                           0.189000
Name: proportion, dtype: float64


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 500       # ~100 words
chunk_overlap = 50     # ensure context continuity

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    add_start_index=True,
)

def create_chunks(df):
    chunked_docs = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Chunking narratives"):
        chunks = text_splitter.split_text(row['cleaned_narrative'])

        for i, chunk in enumerate(chunks):
            chunked_docs.append({
                "page_content": chunk,
                "metadata": {
                    "complaint_id": row['Complaint ID'],
                    "product": row['Product'],
                    "company": row['Company'],
                    "state": row.get('State', ''),
                    "date_received": row.get('Date received', ''),
                    "word_count": len(chunk.split()),
                    "chunk_index": i
                }
            })
    return chunked_docs

docs = create_chunks(df_sample)
print(f"Created {len(docs)} total chunks from {len(df_sample)} complaints.")

/Users/mac/Documents/kaim/Rag_complaint_chatbot/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Chunking narratives: 100%|██████████| 15000/15000 [00:11<00:00, 1257.88it/s]

Created 41434 total chunks from 15000 complaints.


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

# Automatically use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': False}
)

Using device: cpu


In [4]:
from langchain_community.vectorstores import Chroma

persist_directory = '../vector_store/complaints_db'
os.makedirs(persist_directory, exist_ok=True)

texts = [doc['page_content'] for doc in docs]
metadatas = [doc['metadata'] for doc in docs]

print("Starting vector store indexing... This may take 5-10 minutes depending on CPU/GPU.")

vector_db = Chroma.from_texts(
    texts=texts,
    embedding=embeddings,
    metadatas=metadatas,
    persist_directory=persist_directory
)

print(f"✅ Vector Store saved at {persist_directory}")

Starting vector store indexing... This may take 5-10 minutes depending on CPU/GPU.
✅ Vector Store saved at ../vector_store/complaints_db


In [6]:
query = "How do I dispute an unauthorized credit card charge?"
results = vector_db.similarity_search(query, k=3)

if results:
    for i, res in enumerate(results):
        print(f"\nResult {i+1}:")
        print(f"Product: {res.metadata['product']}")
        print(f"Company: {res.metadata['company']}")
        print(f"Chunk Index: {res.metadata['chunk_index']}")
        print(f"Content Preview: {res.page_content[:250]}...")
else:
    print("No results found for query.")


Result 1:
Product: Credit card or prepaid card
Company: UNITED SERVICES AUTOMOBILE ASSOCIATION
Chunk Index: 6
Content Preview: for approimately minutes while someone worked on my issue was told that this is an issue for my bank to fi and was hung up on continues to tell me that i need to call my bank and that because the charges are fraud and im not actually being charged it...

Result 2:
Product: Credit card or prepaid card
Company: UNITED SERVICES AUTOMOBILE ASSOCIATION
Chunk Index: 6
Content Preview: for approimately minutes while someone worked on my issue was told that this is an issue for my bank to fi and was hung up on continues to tell me that i need to call my bank and that because the charges are fraud and im not actually being charged it...

Result 3:
Product: Credit card or prepaid card
Company: Atlanticus Services Corporation
Chunk Index: 0
Content Preview: i had an issue with this company beginning in i used my card to purchase something online i then received a bill wi